In [ ]:
import time
import gc
import os
import torch
import pandas as pd
from huggingface_hub import login
from datasets import load_dataset
from transformers import BitsAndBytesConfig
from transformers import EarlyStoppingCallback
from peft import prepare_model_for_kbit_training
from langchain_community.vectorstores import FAISS
from transformers import TrainingArguments, Trainer
from langchain_core.documents.base import Document
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_community.embeddings import HuggingFaceBgeEmbeddings



# Replace with your actual Hugging Face token


# Load the tokenizer and model
model_name = "meta-llama/Llama-3.1-8B-Instruct"
#model_name='microsoft/phi-2'
#model_name='microsoft/phi-4'
#model_name='microsoft/Phi-3.5-mini-instruct'
#model_name='meta-llama/Llama-3.2-3B'

train = pd.read_excel("/home/ml/-------/ollamawork/Finetuning_2025/Final_training_dataset_bhusan.xlsx")
val = pd.read_excel("/home/ml/-------/ollamawork/Finetuning_2025/Final_test_data_bhusan.xlsx")


train_dataset = train.iloc[:, :]
test_dataset = val.iloc[:, :]

col = {'Test Case Name' : 'name', 'Steps with Expected Output': 'steps', 'Python Code': 'python code', 'Summary': 'summary'}
train_dataset.rename(columns = col, inplace=True)
test_dataset.rename(columns = col, inplace=True)

train_dataset.to_csv('train.csv', index = False)
test_dataset.to_csv('test.csv', index = False)


data_files = {"train": "train.csv", "test": "test.csv"}
dataset = load_dataset("csv", data_files=data_files)



# Setting up the tokenizer for Phi-2
tokenizer=AutoTokenizer.from_pretrained(
    model_name,
    add_eos_token=True,
    trust_remote_code=False
)

tokenizer.pad_token=tokenizer.eos_token
tokenizer.truncation_side="left"





def collate_and_tokenize(examples):
    batch_encoded = []
    labels = []
    for i, (name, steps, answer, summary) in enumerate(zip(examples['name'], examples["steps"], examples["python code"], examples["summary"])):
        prompt = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Generate a complete Python code based on the following test case summary and test steps. Use chain of thought while generating the output.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Name: {name}\n
Test Case: {summary}\n
Test Steps: {steps}\n
Python Code:{answer}\n 
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
        """
        encoded = tokenizer(
            prompt,
            return_tensors="np",
            padding="max_length",
            truncation=True,
            max_length=6500)

        # Flatten input_ids and set labels
        input_ids = encoded["input_ids"].squeeze(0).tolist()
        batch_encoded.append(input_ids)
        labels.append(input_ids)  # Set labels equal to input_ids for language modeling

    # Pad both input_ids and labels to ensure they have uniform length
    padded_batch = tokenizer.pad({"input_ids": batch_encoded, "labels": labels}, padding=True, return_tensors="np")

    return padded_batch

# Tokenize the training and test datasets
columns_to_remove = ["name", "steps", "python code", "summary"]

tokenized_dataset_train = dataset['train'].map(
    collate_and_tokenize,
    batched=True,
    batch_size=1,
    remove_columns=columns_to_remove
)

tokenized_dataset_test = dataset['test'].map(
    collate_and_tokenize,
    batched=True,
    batch_size=1,
    remove_columns=columns_to_remove
)


os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.cuda.empty_cache()

print(torch.cuda.is_available())  # Should output: True
print(torch.cuda.get_device_name())



bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=False
)

from accelerate import init_empty_weights
from transformers import AutoModelForCausalLM

# Use the auto device map for proper placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda:0",  # Automatically maps layers to devices
#device_map={'':torch.xpu.current_device()},
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    trust_remote_code=False,
    low_cpu_mem_usage=True
)





def print_trainable_parameters(model):
    trainable_params=0
    all_params=0
    for _, param in model.named_parameters():
        all_params+=param.numel()
        if param.requires_grad:
            trainable_params+=param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_params} || trainable%: {100 * trainable_params/all_params:.2f}")

print_trainable_parameters(model)

model.get_memory_footprint()

model.config.quantization_config


#gradient checkpointing to save memory
model.gradient_checkpointing_enable()
model.get_memory_footprint()

prepared_model=prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True
)
prepared_model.get_memory_footprint()


peft_config=LoraConfig(
    r=72,
    lora_alpha=144,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj'
    ],
    bias="none",
    lora_dropout=0.05,
#    init_lora_weights=False,
    task_type=TaskType.CAUSAL_LM,
    modules_to_save= None
)
#prepared_model.add_adapter(peft_config, adapter_name="adapter_2")
lora_model=get_peft_model(prepared_model, peft_config)
lora_model.get_memory_footprint()

def print_trainable_parameters(model):
    trainable_params=0
    all_params=0
    for _, param in model.named_parameters():
        all_params+=param.numel()
        if param.requires_grad:
            trainable_params+=param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_params} || trainable%: {100 * trainable_params/all_params:.2f}")

print_trainable_parameters(lora_model)


gc.collect()
torch.cuda.empty_cache()



training_args=TrainingArguments(
    output_dir='/home/ml/-------/ollamawork/Finetuning_2025/Finetune_llama3.2/Result_llama-3.1_1',
    overwrite_output_dir=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=5,
    gradient_checkpointing=True,  # Enable gradient checkpointing
    gradient_checkpointing_kwargs={"use_reentrant": False},
    warmup_steps=100,
    max_steps=430, # Total number of training steps
    #num_train_epochs=2, # Number of training epochs
    learning_rate=5e-04, # Learning rate
    weight_decay=0.01, # Weight decay
    optim="adamw_8bit", # Keep the optimizer state and quantize it
     bf16=True, # Do not supported in Kaggle environment, require Ampere....
#    fp16=True, # use fp16 16bit(mixed) precision training instead of 32-bit training.
    logging_dir='./logs',
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=4, # Limit the total number of checkpoints
    eval_strategy="steps",
    eval_steps=10,
    load_best_model_at_end=True, # Load the best model at the end of training,
    max_grad_norm=1.0
)

#lora_model.config.use_cache=False
lora_model.config_dropout=0.1
lora_model.config.use_cache=False

trainer=Trainer(
    model=lora_model,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_test,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

start_time=time.time()
trainer.train(resume_from_checkpoint="/home/ml/-------/ollamawork/Finetuning_2025/Finetune_llama3.2/Result_llama-3.1_1/checkpoint-240")
#trainer.train()
end_time=time.time()

training_time=end_time-start_time

print(f"Training completed in {training_time} seconds.")
trainer.save_model("/home/ml/-------/ollamawork/Finetuning_2025/Finetune_llama3.2/Result_llama-3.1_1")
tokenizer.save_pretrained("/home/ml/-------/ollamawork/Finetuning_2025/Finetune_llama3.2/Result_llama-3.1_1")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer
from datasets import load_dataset
import pandas as pd
from huggingface_hub import login
from datasets import load_dataset
from transformers import BitsAndBytesConfig
from transformers import AutoModelForCausalLM, AutoConfig
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
val = pd.read_excel("/home/ml/-------/ollamawork/Finetuning_2025/Final_test_data_bhusan.xlsx")

#model_name='/home/ml/-------/ollamawork/Finetuning_2025/Finetune_phi4/Results/Result_phi4_8'
#model_name = "deepseek-ai/deepseek-coder-6.7b-instruct"

# Load the tokenizer and model
model_name = '/home/ml/-------/ollamawork/Finetuning_2025/Finetune_llama3.2/Result_llama-3.1_1'



        # Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=False
)

from accelerate import init_empty_weights
from transformers import AutoModelForCausalLM

# Load the fine-tuned model
model = AutoModelForCausalLM.from_pretrained(model_name,  torch_dtype=torch.bfloat16, device_map="auto")

# Set the padding token to eos_token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = "left" 


def gen(name, summary, steps):
    new_prompt = f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Generate a complete Python code based on the following test case summary and test steps. Use chain of thought while generating the output.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Name: {name}\n
Test Case: {summary}\n
Test Steps: {steps}\n
Python Code:\n 
        """
    # Tokenize the input
    inputs = tokenizer(
        new_prompt,
        return_tensors="pt",
        return_attention_mask=True,  # Enable attention mask
        padding=True,
        truncation=True
    )

    # Move inputs to GPU
    inputs = {key: value.to('cuda') for key, value in inputs.items()}
    torch.manual_seed(42)

    # Generate text
    outputs = model.generate(
        **inputs,
        max_new_tokens=6500,  # Limit length to prevent excessive output
        eos_token_id=tokenizer.eos_token_id,  # Set end-of-sequence token
        #repetition_penalty=1,  # Discourage repetitive generation
      #  top_k=100,  # Limit sampling to the top 50 tokens
       # top_p=0.7, #0.9 very bad
        temperature=0.1,
        do_sample=True
    )
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return generated_text

n=4
#i=0
with open('/home/ml/-------/ollamawork/Finetuning_2025/Finetune_llama3.2/Result_llama3.1-1_0.1-output.txt','a') as file:
      for i in range(len(val)-n): 
        
        file.write("\n\n\n\n\n")
        # Get the generated content
        content = gen( val['Test Case Name'][i+n], val['Summary'][i+n], val['Steps with Expected Output'][i+n])
        
        # Check if content is None and handle it
        if content is None:
            content = ""  # Default to an empty string or some placeholder text
        file.write(content)
        
        file.write("\n\n\n\n\n")
        file.write('--------------------------------------------------------------------------------------------------------\n')
        print(str(i+1+n) + " is done.")

